In [ ]:
# Celda 1 · Importaciones
# En Colab TensorFlow ya viene instalado: no hace falta pip install.
import math

import numpy as np
import tensorflow as tf
from tensorflow.keras import Input, Model, layers

print("TensorFlow:", tf.__version__)


In [ ]:
# Celda 2 · Discriminador PatchGAN condicional
# Contrato:
#   entradas : x (foto degradada) e y (objetivo real o G(x)), ambas (B, 256, 256, 3) en [-1, 1]
#   salida   : mapa de logits (B, 30, 30, 1), SIN sigmoide
# Cada uno de los 30 x 30 valores juzga un parche de 70 x 70 píxeles de la entrada.

def _bloque_conv(x, filtros, stride, inicializador, usar_norm=True):
    """Bloque básico: Conv 4x4 -> (BatchNorm) -> LeakyReLU(0.2).

    Con padding="same" y stride=2 la imagen se reduce a la mitad
    (256 -> 128 -> 64 -> 32).
    """
    x = layers.Conv2D(
        filtros,
        kernel_size=4,
        strides=stride,
        padding="same",
        kernel_initializer=inicializador,
        # Si después viene BatchNorm el sesgo (bias) sobra, porque BatchNorm
        # ya tiene su propio desplazamiento aprendible.
        use_bias=not usar_norm,
    )(x)

    if usar_norm:
        # Normaliza cada canal para que el entrenamiento sea más estable.
        x = layers.BatchNormalization()(x)

    # LeakyReLU deja pasar un poco de señal negativa (pendiente 0.2). Así el
    # discriminador no "apaga" neuronas y siempre hay gradiente hacia G.
    return layers.LeakyReLU(0.2)(x)


def build_discriminator(image_size: int = 256, ndf: int = 64) -> Model:
    """Construye el discriminador PatchGAN condicional.

    Args:
        image_size: lado de la imagen cuadrada (clave image_size de base.yaml).
        ndf: número de filtros de la primera capa (clave ndf de base.yaml).
            Las capas siguientes lo multiplican por 2, 4 y 8.

    Returns:
        Un modelo Keras con dos entradas [x, y] y una salida de logits con
        forma (B, 30, 30, 1) cuando image_size = 256.
    """
    # Pesos iniciales ~ Normal(0, 0.02), como en el paper de Pix2Pix.
    inicializador = tf.random_normal_initializer(0.0, 0.02)

    # 1) Dos entradas: la foto degradada (la condición) y la imagen a juzgar.
    entrada = Input(shape=(image_size, image_size, 3), name="degradada")
    objetivo = Input(shape=(image_size, image_size, 3), name="real_o_generada")

    # 2) Las unimos por canales: 3 + 3 = 6 canales. Así el discriminador ve
    #    la condición y puede decidir si y es una restauración coherente CON
    #    ESA x, no solo si y "parece una foto real" (por eso es condicional).
    x = layers.Concatenate()([entrada, objetivo])  # (B, 256, 256, 6)

    # 3) Tres bloques con stride 2: cada uno reduce el tamaño a la mitad.
    #    El primero no lleva BatchNorm (igual que en el paper).
    x = _bloque_conv(x, ndf, 2, inicializador, usar_norm=False)  # (B,128,128, 64)
    x = _bloque_conv(x, ndf * 2, 2, inicializador)  # (B, 64, 64, 128)
    x = _bloque_conv(x, ndf * 4, 2, inicializador)  # (B, 32, 32, 256)

    # 4) Cuarto bloque con stride 1. Aquí NO usamos padding="same" porque
    #    queremos que el tamaño baje un poco: 1 de relleno a cada lado (34) y
    #    una convolución 4x4 sin relleno da 34 - 4 + 1 = 31.
    x = layers.ZeroPadding2D(padding=1)(x)  # (B, 34, 34, 256)
    x = layers.Conv2D(
        ndf * 8,
        kernel_size=4,
        strides=1,
        kernel_initializer=inicializador,
        use_bias=False,
    )(x)  # (B, 31, 31, 512)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(0.2)(x)

    # 5) Capa final: un solo filtro y SIN activación. Devuelve logits, no
    #    probabilidades: la sigmoide la aplica la pérdida (BCE con logits),
    #    que es más estable numéricamente. 33 - 4 + 1 = 30.
    x = layers.ZeroPadding2D(padding=1)(x)  # (B, 33, 33, 512)
    logits = layers.Conv2D(
        1,
        kernel_size=4,
        strides=1,
        kernel_initializer=inicializador,
        name="logits_parches",
    )(x)  # (B, 30, 30, 1)

    return Model(
        inputs=[entrada, objetivo],
        outputs=logits,
        name="patchgan_discriminator",
    )


In [ ]:
# Celda 3 · Pérdidas de Pix2Pix
#   L_D = 0.5 * [ BCE(D(x, y), 1) + BCE(D(x, G(x)), 0) ]
#   L_G = BCE(D(x, G(x)), 1) + lambda * L1(G(x), y)      con lambda = 100
# discriminator_loss y generator_loss devuelven un ESCALAR.

# BCE "con logits": aplica la sigmoide y la entropía cruzada en un solo paso.
# Es más estable que hacer sigmoide + BCE por separado (evita log(0) y los
# gradientes pobres). Promedia sobre el batch y sobre los 30 x 30 parches.
_bce = tf.keras.losses.BinaryCrossentropy(from_logits=True)


def adversarial_loss(logits, target_is_real):
    """BCE entre los logits de D y la etiqueta que queremos (1 real, 0 falsa).

    Args:
        logits: salida de D, forma (B, 30, 30, 1).
        target_is_real: True si los parches deberían clasificarse como reales.
    """
    # Una etiqueta por parche: un mapa de unos o de ceros con la forma de los
    # logits (tf.ones_like / tf.zeros_like copian la forma y el tipo).
    if target_is_real:
        etiquetas = tf.ones_like(logits)
    else:
        etiquetas = tf.zeros_like(logits)
    return _bce(etiquetas, logits)


def l1_loss(target, gen_output):
    """Error absoluto medio entre el objetivo real y lo generado.

    Es la parte que obliga al generador a ser FIEL a la imagen original.
    Se usa L1 y no L2 porque L2 tiende a promediar y deja imágenes borrosas.
    """
    return tf.reduce_mean(tf.abs(target - gen_output))


def discriminator_loss(real_logits, fake_logits):
    """Pérdida de D: debe dar 1 a lo real y 0 a lo generado.

    Args:
        real_logits: D(x, y), lo que D opina de la imagen real.
        fake_logits: D(x, G(x)), lo que D opina de la imagen generada.

    Returns:
        Escalar. El factor 0.5 hace que D aprenda más lento que G, como en
        el artículo original de Pix2Pix.
    """
    perdida_real = adversarial_loss(real_logits, target_is_real=True)
    perdida_falsa = adversarial_loss(fake_logits, target_is_real=False)
    return 0.5 * (perdida_real + perdida_falsa)


def generator_loss(fake_logits, gen_output, target, lambda_l1=100.0):
    """Pérdida total de G: engañar a D + ser fiel al objetivo (L1).

    Args:
        fake_logits: D(x, G(x)), lo que D opina de la imagen generada.
        gen_output: G(x), la imagen restaurada, (B, 256, 256, 3).
        target: y, la imagen original sin degradar, (B, 256, 256, 3).
        lambda_l1: peso del L1 (clave lambda_l1 de base.yaml). Multiplica
            SOLO al L1, nunca a la pérdida adversarial.

    Returns:
        Escalar: adversarial + lambda_l1 * L1.
    """
    # Para G queremos que D diga "real" (1) sobre lo generado: por eso la
    # etiqueta es 1 aunque la imagen sea falsa ("non-saturating loss").
    adversarial = adversarial_loss(fake_logits, target_is_real=True)
    l1 = l1_loss(target, gen_output)
    return adversarial + lambda_l1 * l1


In [ ]:
# Celda 4 · Prueba del discriminador
discriminador = build_discriminator(image_size=256, ndf=64)

# Dos imágenes aleatorias en [-1, 1], como las que entrega el generador.
x = tf.random.uniform((2, 256, 256, 3), -1.0, 1.0, seed=1)
y = tf.random.uniform((2, 256, 256, 3), -1.0, 1.0, seed=2)
salida = discriminador([x, y], training=False)

# Contrato: la salida debe ser (B, 30, 30, 1).
assert salida.shape == (2, 30, 30, 1), salida.shape
assert np.isfinite(salida.numpy()).all()
# Última capa sin activación: devuelve logits, no probabilidades.
assert discriminador.get_layer("logits_parches").get_config()["activation"] == "linear"
# Es condicional: si cambia x (con la misma y), la salida cambia.
otra_x = tf.random.uniform((2, 256, 256, 3), -1.0, 1.0, seed=99)
assert not np.allclose(salida.numpy(), discriminador([otra_x, y], training=False).numpy())

print("Forma de salida:", salida.shape)
print("Parámetros:", f"{discriminador.count_params():,}")
discriminador.summary()


In [ ]:
# Celda 5 · Prueba de las pérdidas con valores conocidos
# Con logits = 0 la sigmoide da 0.5 y BCE = -ln(0.5) = ln(2), con etiqueta 0 o 1:
# es el valor de un discriminador "indeciso".
LN2 = math.log(2.0)


def logits(valor):  # mapa de logits constante con la forma de la salida de D
    return tf.fill((2, 30, 30, 1), float(valor))


def imagen(valor):  # imagen constante con la forma de las del generador
    return tf.fill((2, 256, 256, 3), float(valor))


# 1) BCE con logits = 0 vale ln(2), sea la etiqueta real o falsa.
for es_real in (True, False):
    assert math.isclose(float(adversarial_loss(logits(0), es_real)), LN2, rel_tol=1e-5)

# 2) D indeciso -> L_D = ln(2).
assert math.isclose(float(discriminator_loss(logits(0), logits(0))), LN2, rel_tol=1e-5)

# 3) D perfecto (real -> +20, falsa -> -20) -> pérdida casi 0.
assert float(discriminator_loss(logits(20), logits(-20))) < 1e-6

# 4) D totalmente equivocado -> pérdida ~20.
assert math.isclose(float(discriminator_loss(logits(-20), logits(20))), 20.0, rel_tol=1e-3)

# 5) L1: |0.2 - (-0.3)| = 0.5 en todos los píxeles -> media 0.5.
assert math.isclose(float(l1_loss(imagen(0.2), imagen(-0.3))), 0.5, rel_tol=1e-6)

# 6) Sin error L1 solo queda la parte adversarial (ln 2).
assert math.isclose(
    float(generator_loss(logits(0), imagen(0.1), imagen(0.1))), LN2, rel_tol=1e-5
)

# 7) Lambda multiplica SOLO el L1: con L1 = 0.5 y lambda = 100 -> ln2 + 50; con lambda = 0 -> ln2.
gen, objetivo = imagen(0.0), imagen(0.5)
con_100 = float(generator_loss(logits(0), gen, objetivo, lambda_l1=100.0))
con_0 = float(generator_loss(logits(0), gen, objetivo, lambda_l1=0.0))
assert math.isclose(con_100, LN2 + 50.0, rel_tol=1e-5)
assert math.isclose(con_0, LN2, rel_tol=1e-5)

# 8) Contrato con train.py: ambas pérdidas son escalares float32.
for perdida in (
    discriminator_loss(logits(0), logits(0)),
    generator_loss(logits(0), imagen(0.0), imagen(0.0)),
):
    assert perdida.shape == () and perdida.dtype == tf.float32

# 9) Logits extremos: sin NaN ni infinitos.
for extremo in (-1e4, 1e4):
    assert np.isfinite(float(discriminator_loss(logits(extremo), logits(-extremo))))
    assert np.isfinite(float(generator_loss(logits(extremo), imagen(1.0), imagen(-1.0))))

print("✅ Todas las pruebas de pérdidas pasaron")
print(f"   L_D indeciso = {float(discriminator_loss(logits(0), logits(0))):.4f} (ln 2 = {LN2:.4f})")
print(f"   L_G con lambda=100 y L1=0.5 = {con_100:.4f} (ln 2 + 50 = {LN2 + 50:.4f})")


In [ ]:
# Celda 6 · Generador U-Net (rol B). Equivale a blocks.py + generator.py del repo.
# Contrato: (B, H, W, 3) -> (B, H, W, 3) en [-1, 1] (tanh), con H = W = 64, 128 o 256.
from tensorflow.keras import initializers

# Normalizaciones aceptadas. None significa "sin normalización".
NORMALIZACIONES = ("instance", "batch", None)


def crear_inicializador():
    """Pesos iniciales ~ Normal(0, 0.02), como en el paper de Pix2Pix."""
    return initializers.RandomNormal(mean=0.0, stddev=0.02)


def normalizar(x, tipo):
    """Aplica la normalización elegida.

    - "instance": normaliza cada imagen y cada canal por separado. No depende del
      tamaño del lote, por eso es estable con lotes de 1 a 4 imágenes (BatchNorm no).
      En Keras es GroupNormalization con un grupo por canal (groups=-1).
    - "batch": BatchNormalization clásica.
    - None: no hace nada.
    """
    if tipo not in NORMALIZACIONES:
        raise ValueError(f"norm debe ser uno de {NORMALIZACIONES}, no {tipo!r}")
    if tipo == "instance":
        return layers.GroupNormalization(groups=-1)(x)
    if tipo == "batch":
        return layers.BatchNormalization()(x)
    return x


def bloque_encoder(x, filtros, inicializador, norm="instance"):
    """Encoder: Conv 4x4 stride 2 -> (normalización) -> LeakyReLU(0.2).

    El stride 2 reduce el alto y el ancho a la mitad (256 -> 128 -> 64 ...).
    """
    x = layers.Conv2D(
        filtros,
        kernel_size=4,
        strides=2,
        padding="same",
        kernel_initializer=inicializador,
        # Si hay normalización, el sesgo sobra: la normalización ya tiene el suyo.
        use_bias=(norm is None),
    )(x)
    x = normalizar(x, norm)
    # LeakyReLU deja pasar un poco de señal negativa y evita neuronas "muertas".
    return layers.LeakyReLU(0.2)(x)


def bloque_decoder(x, filtros, inicializador, norm="instance", dropout=0.0):
    """Decoder: ConvTranspose 4x4 stride 2 -> normalización -> (Dropout) -> ReLU.

    El stride 2 duplica el alto y el ancho (1 -> 2 -> 4 ...). El dropout se usa
    solo en los primeros bloques del decoder: mete ruido que reemplaza al vector z
    de las GAN clásicas y evita que el generador memorice.
    """
    x = layers.Conv2DTranspose(
        filtros,
        kernel_size=4,
        strides=2,
        padding="same",
        kernel_initializer=inicializador,
        use_bias=(norm is None),
    )(x)
    x = normalizar(x, norm)
    if dropout > 0:
        x = layers.Dropout(dropout)(x)
    return layers.ReLU()(x)


# Máximo de reducciones a la mitad: 256 -> 1 necesita 8. Más niveles no caben en GPUs
# pequeñas y no aportan para imágenes de 256 x 256.
MAX_NIVELES = 8


def niveles_por_defecto(image_size):
    """Cuántas veces hay que dividir entre 2 para llegar a 1 x 1 (tope: 8).

    256 -> 8, 128 -> 7, 64 -> 6.
    """
    return min(MAX_NIVELES, image_size.bit_length() - 1)


def build_generator(image_size=256, ngf=64, niveles=None, norm="instance", dropout=0.5):
    """Construye el generador U-Net.

    Args:
        image_size: lado de la imagen cuadrada (clave image_size de base.yaml).
        ngf: filtros de la primera capa (clave ngf de base.yaml). Los niveles
            siguientes lo duplican hasta un máximo de 8 * ngf.
        niveles: número de bloques del encoder. Por defecto, los necesarios para
            llegar a 1 x 1. image_size debe ser divisible por 2 ** niveles.
        norm: "instance" (por defecto), "batch" o None.
        dropout: tasa de dropout de los 3 primeros bloques del decoder (0 = sin dropout).

    Returns:
        Modelo Keras con una entrada (B, image_size, image_size, 3) y una salida
        con la misma forma, en [-1, 1].
    """
    if niveles is None:
        niveles = niveles_por_defecto(image_size)
    if not 1 <= niveles <= MAX_NIVELES:
        raise ValueError(f"niveles debe estar entre 1 y {MAX_NIVELES}, no {niveles}")
    if image_size % (2**niveles) != 0:
        raise ValueError(
            f"image_size={image_size} no es divisible por 2**{niveles}={2**niveles}"
        )
    if norm not in NORMALIZACIONES:
        raise ValueError(f"norm debe ser uno de {NORMALIZACIONES}, no {norm!r}")

    inicializador = crear_inicializador()
    entrada = Input(shape=(image_size, image_size, 3), name="entrada_degradada")

    # 1) ENCODER. Cada bloque reduce el tamaño a la mitad y guardamos su salida para
    #    usarla después como skip connection. Los filtros se duplican hasta 8 * ngf.
    #    El primer bloque y el último (cuello de botella) van SIN normalización:
    #    el cuello de botella mide 1 x 1 y InstanceNorm lo dejaría en 0 (con un solo
    #    valor por canal, valor - media = 0), anulando la señal y el gradiente.
    filtros = [min(ngf * 2**i, ngf * 8) for i in range(niveles)]
    x = entrada
    saltos = []
    for i, f in enumerate(filtros):
        sin_norma = i == 0 or i == niveles - 1
        x = bloque_encoder(x, f, inicializador, norm=None if sin_norma else norm)
        saltos.append(x)

    # 2) DECODER con skip connections. La última salida del encoder (cuello de botella)
    #    es el punto de partida; las demás se usan en orden inverso: la más profunda
    #    primero. Cada skip tiene el mismo tamaño que la salida del bloque decoder,
    #    así que se pueden concatenar por canales. Esto lleva los detalles finos del
    #    encoder directo al decoder, sin pasar por el cuello de botella.
    saltos = list(reversed(saltos[:-1]))
    for i, salto in enumerate(saltos):
        x = bloque_decoder(
            x,
            salto.shape[-1],
            inicializador,
            norm=norm,
            dropout=dropout if i < 3 else 0.0,  # dropout solo en los 3 primeros
        )
        x = layers.Concatenate()([x, salto])

    # 3) Último bloque del decoder: recupera el tamaño original. No hay skip porque el
    #    encoder no tiene ninguna salida con la resolución de la entrada.
    x = bloque_decoder(x, ngf, inicializador, norm=norm)

    # 4) Salida: Conv 3x3 a 3 canales con tanh, que limita los valores a [-1, 1]
    #    (el mismo rango en que se normalizan las imágenes del dataset).
    salida = layers.Conv2D(
        3,
        kernel_size=3,
        padding="same",
        kernel_initializer=inicializador,
        activation="tanh",
        name="salida_tanh",
    )(x)

    return Model(inputs=entrada, outputs=salida, name="unet_generator")


In [ ]:
# Celda 7 · Conexión G + D: ¿encajan las formas y los rangos?
generador = build_generator(image_size=256, ngf=64)
discriminador = build_discriminator(image_size=256, ndf=64)

x = tf.random.uniform((2, 256, 256, 3), -1.0, 1.0, seed=1)  # fotos degradadas
y = tf.random.uniform((2, 256, 256, 3), -1.0, 1.0, seed=2)  # objetivos reales

# G: foto degradada -> imagen restaurada.
gen = generador(x, training=False)
assert gen.shape == (2, 256, 256, 3), gen.shape
assert float(tf.reduce_min(gen)) >= -1.0 and float(tf.reduce_max(gen)) <= 1.0  # tanh

# D: juzga (x, y real) y (x, G(x)). Lo que sale de G entra tal cual a D.
logits_reales = discriminador([x, y], training=False)
logits_falsos = discriminador([x, gen], training=False)
assert logits_reales.shape == logits_falsos.shape == (2, 30, 30, 1)

# Las pérdidas aceptan lo que producen G y D.
print("L_D:", float(discriminator_loss(logits_reales, logits_falsos)))
print("L_G:", float(generator_loss(logits_falsos, gen, y)))

print("✅ G y D encajan")
print("Parámetros G:", f"{generador.count_params():,}", "| D:", f"{discriminador.count_params():,}")


In [ ]:
# Celda 8 · Paso de entrenamiento de prueba (datos sintéticos)
# Verifica que el ciclo completo funciona: los gradientes llegan a G y a D y los pesos cambian.
g_opt = tf.keras.optimizers.Adam(2e-4, beta_1=0.5)
d_opt = tf.keras.optimizers.Adam(2e-4, beta_1=0.5)


@tf.function
def paso_de_entrenamiento(x, y):
    # Dos cintas: una registra lo necesario para el gradiente de G y otra para el de D.
    with tf.GradientTape() as cinta_g, tf.GradientTape() as cinta_d:
        gen = generador(x, training=True)  # G restaura
        logits_reales = discriminador([x, y], training=True)  # D mira lo real
        logits_falsos = discriminador([x, gen], training=True)  # D mira lo generado

        g_loss = generator_loss(logits_falsos, gen, y)  # adversarial + 100 * L1
        d_loss = discriminator_loss(logits_reales, logits_falsos)

    grads_g = cinta_g.gradient(g_loss, generador.trainable_variables)
    grads_d = cinta_d.gradient(d_loss, discriminador.trainable_variables)
    g_opt.apply_gradients(zip(grads_g, generador.trainable_variables))
    d_opt.apply_gradients(zip(grads_d, discriminador.trainable_variables))
    # Se devuelven también las partes de L_G para poder mirarlas por separado.
    return g_loss, d_loss, adversarial_loss(logits_falsos, True), l1_loss(y, gen), grads_g, grads_d


peso_antes = discriminador.trainable_variables[0].numpy().copy()

for paso in range(3):
    # Lote sintético: y es x con un poco de ruido, para que el L1 tenga sentido.
    x = tf.random.uniform((2, 256, 256, 3), -1.0, 1.0)
    y = tf.clip_by_value(x + tf.random.normal(x.shape, stddev=0.1), -1.0, 1.0)
    g_loss, d_loss, adv, l1, grads_g, grads_d = paso_de_entrenamiento(x, y)

    # Sin NaN, y todas las variables recibieron gradiente (ninguna quedó desconectada).
    assert np.isfinite([float(g_loss), float(d_loss)]).all()
    assert all(g is not None for g in grads_g) and all(g is not None for g in grads_d)
    print(f"paso {paso + 1}: L_G={float(g_loss):.3f} (adv={float(adv):.3f}, L1={float(l1):.3f}) | L_D={float(d_loss):.3f}")

# Los pesos de D tienen que haber cambiado tras entrenar.
assert not np.allclose(peso_antes, discriminador.trainable_variables[0].numpy())
print("✅ Paso de entrenamiento OK: gradientes a G y D, pérdidas finitas, pesos actualizados")


In [ ]:
# Celda 9 · Lista "Listo cuando" del PR-B2 (64/128/256, tanh, gradientes, parámetros)
# Comprueba el generador contra los criterios del plan del rol B.

# 1) Formas y rango a 64, 128 y 256 (ngf=8 para que corra rápido en CPU).
for lado in (64, 128, 256):
    g = build_generator(image_size=lado, ngf=8)
    salida = g(tf.random.uniform((2, lado, lado, 3), -1.0, 1.0), training=False)
    assert salida.shape == (2, lado, lado, 3), (lado, salida.shape)
    assert float(tf.reduce_min(salida)) >= -1.0 and float(tf.reduce_max(salida)) <= 1.0
    print(f"✅ {lado}x{lado}: forma {tuple(salida.shape)}, rango dentro de [-1, 1]")

# 2) Batch variable: el mismo modelo acepta lotes de 1, 3 y 5 imágenes.
g = build_generator(image_size=64, ngf=8)
for b in (1, 3, 5):
    assert g(tf.zeros((b, 64, 64, 3)), training=False).shape[0] == b
print("✅ batch variable")

# 3) Los gradientes fluyen: TODOS los parámetros reciben gradiente distinto de cero.
for norm in ("instance", "batch"):
    g = build_generator(image_size=64, ngf=8, norm=norm)
    x = tf.random.uniform((2, 64, 64, 3), -1.0, 1.0)
    with tf.GradientTape() as cinta:
        perdida = tf.reduce_mean(tf.abs(g(x, training=True) - x))
    grads = cinta.gradient(perdida, g.trainable_variables)
    sin_grad = [v.name for v, gr in zip(g.trainable_variables, grads) if gr is None]
    en_cero = [v.name for v, gr in zip(g.trainable_variables, grads)
               if gr is not None and float(tf.reduce_max(tf.abs(gr))) == 0.0]
    assert not sin_grad and not en_cero, (norm, sin_grad, en_cero)
    print(f"✅ gradientes con norm={norm}: {len(grads)} variables, ninguna sin actualizar")

# 4) Conteo de parámetros (solo se construye, no se ejecuta).
p_grande = build_generator(image_size=256, ngf=64).count_params()
p_humo = build_generator(image_size=64, ngf=8, niveles=4).count_params()
print(f"Parámetros ngf=64, 8 niveles: {p_grande:,}  (plan del rol B: ~54 M)")
print(f"Parámetros ngf=8,  4 niveles: {p_humo:,}  (el plan dice ~0,7 M: no coincide, ver notas)")
